In [ ]:

# import vari
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb


In [ ]:
# 100k righe occupano ~140mb di ram, tutto il file è decisamente troppo massiccio
# sample = pd.read_csv("./data/data.csv" , nrows=100000)
# sample.info(memory_usage="deep")

samplesize = 0.05

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ genera sample

# rng = np.random.default_rng(42)
# # quindi leggiamo il file a chunk di 100k righe 
# chunks = pd.read_csv("./data/data.csv", chunksize=100_000)
# # e per ogni chunk ne tiriamo fuori un pezzetto casuale, di grandezza un ventesimo, e concateniamo tutto
# sample = pd.concat(chunk.sample(frac=samplesize, random_state=rng) for chunk in chunks)
# # e poi salviamo in un pkl per non dover rivare tutto il giro ogni volta
# sample.to_pickle(f"./data/sample_{samplesize}.pkl")

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ carica sample
sample = pd.read_pickle(f"./data/sample_{samplesize}.pkl")

dataset = sample
dataset.info()


In [ ]:

valuesPeek = pd.DataFrame({
    "dtype": sample.dtypes,
    "dist_vals": sample.nunique(),
    "n_miss": sample.isna().sum(),
    "%_miss": (sample.isna().sum() / len(sample) * 100),
    "values": pd.Series({c: sample[c].value_counts().head(10).index.tolist()
                     for c in sample.columns}),
})

# i caratteri arabi irritano overleaf, quindi quando una stringa non è latin-1, la sostituiamo con un placeholder
filterLatin = lambda s: str(s) if str(s).encode("latin-1", "ignore").decode("latin-1") == str(s) else "<non-latin-string>"
valuesPeek["values"] = valuesPeek["values"].apply(lambda v: ", ".join(map(filterLatin, v)))

# e poi, siccome overleaf regge male anche le sbrodolate, tagliamo la colonna values a un numero ragionevole di caratteri
n = 20
valuesPeek["values"] = valuesPeek["values"].apply(lambda s: s if len(s) <= n else s[:n] + "...")

valuesPeek.to_latex("tables/feature_peek.tex", escape=True, float_format="%.1f",
             column_format="llrrrp")
valuesPeek

# DATA UNDERSTANDING


In [ ]:
# following e follower count hanno valori letterali, vediamo quali sono
fgc = dataset["following_count"]
fgcNumeric = pd.to_numeric(fgc, errors="coerce")

fgcInvalid = fgc[fgcNumeric.isna() & fgc.notna()]
fgcInvalid.value_counts()

frc = dataset["follower_count"]
frcNumeric = pd.to_numeric(frc, errors="coerce")

frcInvalid = frc[frcNumeric.isna() & frc.notna()]
frcInvalid.value_counts()

dataset["following_count_clean"] = fgcNumeric.astype("Int64")
dataset["follower_count_clean"] = frcNumeric.astype("Int64")

In [ ]:
# ci sono dei booleani scritti come stringhe, e un  is_control?, che non si sa se mettere true o false
# anche in questo caso, in una nuova colonna
dataset["is_control"].map(lambda v: (str(v), type(v).__name__)).value_counts()

dataset["is_control_clean"] = dataset["is_control"].map(
    {True: True, "True": True, False: False, "False": False}
).astype("boolean")

In [ ]:
# trasforma da stringa a datetime
dataset["post_time_clean"] = pd.to_datetime(dataset["post_time"], format="mixed", errors="coerce")
# dataset["post_time_clean"].isna().any()

dataset["account_creation_date_clean"] = pd.to_datetime(dataset["account_creation_date"], format="mixed", errors="coerce")
# dataset["account_creation_date_clean"].isna().any()

In [ ]:
# nel post_text le menzioni sono discutibilmente testo semanticamente significativo, ma gonfiano molto la lunghezza
# quindi ha senso eliminarle (e analizzare in seguito) 
dataset["post_text_clean"]= dataset["post_text"].str.replace(r"@\w+", "", regex=True)

# DATA PREPARATION

In [ ]:

# Set up a grid of plots:
fig = plt.figure(figsize=(20, 10)) 
fig_dims = (4, 3)
fig.subplots_adjust(hspace=0.4, wspace=0.4)

plt.subplot2grid(fig_dims, (0, 0))
dataset["ioCountry"].value_counts().plot(kind='bar',title='numero di post per stato')

plt.subplot2grid(fig_dims, (0, 1))
# quando si tratta di punti nel tempo, usare sturges o simili per calcolare il numero di bucket
# sarebbe sbagliato, perché non seguono una distribuzione normale (potrebbero farlo come ora del giorno)
# è più ragionevole usare un numero fisso i buckets che non sia troppo alto da risultare scomodo
# pandas si incasina con le date, quindi prima assegnamo a ogni record il suo bucket
ptc = dataset["post_time_clean"]
buckets = pd.cut(ptc, bins=20).value_counts(sort=False)
# poi sistemiamo l'etichetta da precisione piena a giusto il giorno
buckets.index = [f"{b.left:%Y-%m-%d} – {b.right:%Y-%m-%d}" for b in buckets.index]
# e poi plottiamo
buckets.plot(kind="bar", title="periodo dei post")
# dataset["post_time_clean"].plot(kind="hist", title="periodo dei post", bins=20)

plt.subplot2grid(fig_dims, (0, 2))
# usare la lunghezza del testo è un po' grezzo, così come il numero di parole (possono cambiare col linguaggio),
# ma per farsi un'idea iniziale funziona. Anche in questo caso sturges è un po' grezzo, e usiamo un numero comodo di bins

dataset["post_text_clean"].str.len().plot(kind="hist", bins=30) 
# anche in questo caso ci sono evidentemente degli outlier molto lunghi e poco comuni
